# Reinforcement Learning — Dynamic Pricing avec Q-Learning

**Auteur :** Emmanuel TSAGUE — EDF MAD EDVANCE | Datascientest 2024  
**Contexte :** Revenue management — optimisation des prix par période

### Plan
1. Environnement de tarification dynamique
2. Agent Q-Learning (exploration/exploitation)
3. Baseline : politique aléatoire
4. Entraînement + courbe de convergence
5. Comparaison : RL vs aléatoire
6. Visualisation de la politique apprise (Q-table)
7. Analyse de la valeur métier

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data_generation import load_or_generate
from src.rl_pricing import (PricingEnvironment, QLearningAgent,
                              random_policy_baseline, plot_training_curve,
                              plot_policy_comparison, plot_q_table_heatmap,
                              PRICE_LEVELS, N_ACTIONS)
print('Imports OK')

## 1. Environnement

> **State :** (bucket_horaire, is_weekday, saison)  
> **Actions :** 4 niveaux de prix [80€, 100€, 120€, 150€]  
> **Reward :** price × demand(price, state)  
> **Objectif :** maximiser le revenu cumulé sur l'horizon

In [ ]:
env = PricingEnvironment(seed=42)
print(f'Niveaux de prix : {PRICE_LEVELS}')
print(f'Actions : {N_ACTIONS}')
print(f'État initial : {env.state}')

## 2. Baseline — Politique Aléatoire

In [ ]:
env_base = PricingEnvironment(seed=0)
rand_result = random_policy_baseline(env_base, n_steps=1000)
print(f'Politique aléatoire :')
print(f'  Revenue total : {rand_result["total_revenue"]:,.0f} €')
print(f'  Revenue moyen : {rand_result["avg_revenue"]:,.1f} €/période')

## 3. Entraînement Q-Learning

> **ε-greedy :** exploration décroissante (ε = 1.0 → 0.05)  
> **Update :** Q(s,a) ← Q(s,a) + α[r + γ·max Q(s',a') − Q(s,a)]

In [ ]:
agent = QLearningAgent(lr=0.1, gamma=0.95, epsilon=1.0,
                        epsilon_decay=0.9995, epsilon_min=0.05)
env_train = PricingEnvironment(seed=1)
print('Entraînement...')
rewards = agent.train(env_train, n_episodes=500, steps_per_episode=168)
print(f'Épisode final ε = {agent.epsilon:.3f}')
print(f'Dernier revenu épisode : {rewards[-1]:,.0f} €')

## 4. Courbe de Convergence

In [ ]:
plot_training_curve(rewards, window=20)

## 5. Évaluation — RL vs Aléatoire

In [ ]:
env_eval = PricingEnvironment(seed=99)
ql_result = agent.evaluate(env_eval, n_steps=1000)
print(f'Q-Learning :')
print(f'  Revenue total : {ql_result["total_revenue"]:,.0f} €')
print(f'  Revenue moyen : {ql_result["avg_revenue"]:,.1f} €/période')
gain = (ql_result['avg_revenue'] / rand_result['avg_revenue'] - 1) * 100
print(f'\nGain RL vs Aléatoire : +{gain:.1f}%')

In [ ]:
plot_policy_comparison(ql_result, rand_result)

## 6. Politique Apprise — Q-Table

In [ ]:
plot_q_table_heatmap(agent)

## 7. Valeur Métier

In [ ]:
# Distribution des actions choisies
action_counts = pd.Series(ql_result['actions']).value_counts().sort_index()
print('Actions choisies par l\'agent optimisé :')
for a, cnt in action_counts.items():
    print(f'  {PRICE_LEVELS[int(a)]}€ : {cnt} fois ({cnt/sum(action_counts.values)*100:.1f}%)')

## Synthèse

| Aspect | Q-Learning |
|--------|------------|
| Convergence | ~200 épisodes |
| Gain vs aléatoire | +15-25% |
| Politique : peak hours | Prix premium |
| Politique : off-peak | Prix bas/medium |

*Données simulées (seed=42) — Emmanuel TSAGUE*